# Prompt Tuning 实战

## Step1 导入相关包

In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

## Step2 加载数据集

In [2]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [3]:
ds[:3]

{'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。',
  '朱利叶斯·凯撒，又称尤利乌斯·恺撒（Julius Caesar）是古罗马的政治家、军事家和作家。他于公元前44年3月15日被刺杀。 \n\n根据历史记载，当时罗马元老院里一些参议员联合起来策划了对恺撒的刺杀行动，因为他们担心恺撒的统治将给罗马共和制带来威胁。在公元前44年3月15日（又称“3月的艾达之日”），恺撒去参加元老院会议时，被一群参议员包围并被攻击致死。据记载，他身中23刀，其中一刀最终致命。'],
 'input': ['', '输入：4/16', ''],
 'instruction': ['保持健康的三个提示。', '解释为什么以下分数等同于1/4', '朱利叶斯·凯撒是如何死亡的？']}

## Step3 数据集预处理

In [3]:
tokenizer = AutoTokenizer.from_pretrained("../../model/langboat/bloom-1b4-zh")
tokenizer

BloomTokenizerFast(name_or_path='../../model/langboat/bloom-1b4-zh', vocab_size=46145, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [5]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [6]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [7]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

'Human: 解释为什么以下分数等同于1/4\n输入：4/16\n\nAssistant: 4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

In [8]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

## Step4 创建模型

In [9]:
model = AutoModelForCausalLM.from_pretrained("../../model/langboat/bloom-1b4-zh")

## Prompt tuning

### PEFT Step1 配置文件

In [10]:
from peft import PromptTuningConfig, get_peft_model, TaskType, PromptTuningInit

# Soft Prompt
# config = PromptTuningConfig(task_type=TaskType.CAUSAL_LM, num_virtual_tokens=10)
# config
# Hard Prompt
config = PromptTuningConfig(task_type=TaskType.CAUSAL_LM,
                            prompt_tuning_init=PromptTuningInit.TEXT,
                            prompt_tuning_init_text="下面是一段人与机器人的对话。",
                            num_virtual_tokens=len(tokenizer("下面是一段人与机器人的对话。")["input_ids"]),
                            tokenizer_name_or_path="../../model/langboat/bloom-1b4-zh")
config

PromptTuningConfig(peft_type=<PeftType.PROMPT_TUNING: 'PROMPT_TUNING'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, num_virtual_tokens=8, token_dim=None, num_transformer_submodules=None, num_attention_heads=None, num_layers=None, prompt_tuning_init=<PromptTuningInit.TEXT: 'TEXT'>, prompt_tuning_init_text='下面是一段人与机器人的对话。', tokenizer_name_or_path='../../model/langboat/bloom-1b4-zh', tokenizer_kwargs=None)

### PEFT Step2 创建模型

In [11]:
model = get_peft_model(model, config)

In [12]:
model

PeftModelForCausalLM(
  (base_model): BloomForCausalLM(
    (transformer): BloomModel(
      (word_embeddings): Embedding(46145, 2048)
      (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
      (h): ModuleList(
        (0-23): 24 x BloomBlock(
          (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (self_attention): BloomAttention(
            (query_key_value): Linear(in_features=2048, out_features=6144, bias=True)
            (dense): Linear(in_features=2048, out_features=2048, bias=True)
            (attention_dropout): Dropout(p=0.0, inplace=False)
          )
          (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
          (mlp): BloomMLP(
            (dense_h_to_4h): Linear(in_features=2048, out_features=8192, bias=True)
            (gelu_impl): BloomGelu()
            (dense_4h_to_h): Linear(in_features=8192, out_features=2048, bias=True)
          )
        )
      )

In [13]:
model.print_trainable_parameters()

trainable params: 16,384 || all params: 1,303,128,064 || trainable%: 0.0013


## Step5 配置训练参数

In [14]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=10,
    num_train_epochs=1,
    save_steps=2000,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## Step6 创建训练器

In [15]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

## Step7 模型训练

In [16]:
trainer.train()

  0%|          | 0/3357 [00:00<?, ?it/s]

{'loss': 2.9732, 'grad_norm': 1.6640654802322388, 'learning_rate': 4.985105749180816e-05, 'epoch': 0.0}
{'loss': 3.1365, 'grad_norm': 1.5674724578857422, 'learning_rate': 4.9702114983616324e-05, 'epoch': 0.01}
{'loss': 2.9594, 'grad_norm': 0.9389559030532837, 'learning_rate': 4.9553172475424484e-05, 'epoch': 0.01}
{'loss': 2.938, 'grad_norm': 1.7718446254730225, 'learning_rate': 4.940422996723265e-05, 'epoch': 0.01}
{'loss': 2.9647, 'grad_norm': 1.7694813013076782, 'learning_rate': 4.925528745904081e-05, 'epoch': 0.01}
{'loss': 3.0108, 'grad_norm': 1.2461713552474976, 'learning_rate': 4.910634495084897e-05, 'epoch': 0.02}
{'loss': 2.8142, 'grad_norm': 1.4142078161239624, 'learning_rate': 4.895740244265713e-05, 'epoch': 0.02}
{'loss': 2.7301, 'grad_norm': 1.0646862983703613, 'learning_rate': 4.88084599344653e-05, 'epoch': 0.02}
{'loss': 2.741, 'grad_norm': 2.720686912536621, 'learning_rate': 4.865951742627346e-05, 'epoch': 0.03}
{'loss': 2.6354, 'grad_norm': 1.2627583742141724, 'learnin

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 2.4141, 'grad_norm': 10.091264724731445, 'learning_rate': 2.0062555853440572e-05, 'epoch': 0.6}
{'loss': 2.402, 'grad_norm': 4.779909610748291, 'learning_rate': 1.9913613345248736e-05, 'epoch': 0.6}
{'loss': 2.4999, 'grad_norm': 4.645906925201416, 'learning_rate': 1.9764670837056897e-05, 'epoch': 0.6}
{'loss': 2.3517, 'grad_norm': 3.7166378498077393, 'learning_rate': 1.961572832886506e-05, 'epoch': 0.61}
{'loss': 2.547, 'grad_norm': 4.086718559265137, 'learning_rate': 1.946678582067322e-05, 'epoch': 0.61}
{'loss': 2.4227, 'grad_norm': 6.8614702224731445, 'learning_rate': 1.9317843312481382e-05, 'epoch': 0.61}
{'loss': 2.3555, 'grad_norm': 4.3170084953308105, 'learning_rate': 1.9168900804289542e-05, 'epoch': 0.62}
{'loss': 2.4401, 'grad_norm': 4.807779312133789, 'learning_rate': 1.9019958296097706e-05, 'epoch': 0.62}
{'loss': 2.3695, 'grad_norm': 4.965944766998291, 'learning_rate': 1.8871015787905867e-05, 'epoch': 0.62}
{'loss': 2.4374, 'grad_norm': 5.6394476890563965, 'learnin

/home/mizuki/anaconda3/envs/trans/lib/python3.9/site-packages/peft/utils/save_and_load.py:195: UserWarning: Could not find a config file in ../../model/langboat/bloom-1b4-zh - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=3357, training_loss=2.5111096222769267, metrics={'train_runtime': 3111.8723, 'train_samples_per_second': 8.631, 'train_steps_per_second': 1.079, 'total_flos': 1.476411489067008e+16, 'train_loss': 2.5111096222769267, 'epoch': 0.9999255342914588})

## 加载训练好的PEFT模型

In [2]:
from peft import PeftModel

In [3]:
# 在一个jupyter文件中，如果前面已经加载了模型，并对模型做了一定修改，则需要重新加载原始模型
model = AutoModelForCausalLM.from_pretrained("../../model/langboat/bloom-1b4-zh")
peft_model = PeftModel.from_pretrained(model=model, model_id="./chatbot/checkpoint-3357/")

## Step8 模型推理

### 原模型

In [6]:
model = AutoModelForCausalLM.from_pretrained("../../model/langboat/bloom-1b4-zh")
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
print(tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True))

Human: 考试有哪些技巧？

Assistant: 不同的人考试会花费多久的时间呢？
Student: 30-50 秒钟
Assistant: 就那么快吗？
Assistant: 如果用英语答题，时间会更短
Student: 会，但是英语太复杂，很难记忆
Assistant: 我们有专门复习英语的课（课），是2.5 小时
Assistant: 复习英语可以提高成绩
Student: 老师给我们买了很多复习书。
Assistant: 可以买很多课外书籍，这些书可以提高成绩
Assistant: 没有办法让学生自己去


### 微调模型

In [8]:
peft_model = peft_model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(peft_model.device)
print(tokenizer.decode(peft_model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True))

Human: 考试有哪些技巧？

Assistant: 学习技巧主要包括：①要持之以恒，做题之前要养成时间记录的习惯;②学会合理分配学习时间，将每次的考试安排在固定的时间内完成，不要分心;③在平时可以多积累一些复习的材料，遇到不清楚或者不熟悉的地方可以事先多做几遍，了解一下考题的类型和题型，对答题有信心;④在考试前可以多做一些模拟题，掌握一些答题的基本技巧;⑤做题过程中要保持良好的心态，在考试前可以多花一些
